## Review one image

In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt


# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
pred_json_path = Path("/media/gisense/xihan/250812_tamu_cybertraining_team4/PolyWorld/sample_test/tiled_predictions_v8.json")
image_path = Path("/media/data/building_instance_tamu/test/images/hurricane-harvey_00000023_pre_disaster.png")


# ----------------------------------------------------------
# Load original image
# ----------------------------------------------------------
original_img = Image.open(image_path).convert("RGB")
W, H = original_img.size

print("Loaded image:", image_path)
print("Image size:", W, H)

# Create overlay drawing surface
overlay = original_img.copy()
draw = ImageDraw.Draw(overlay)


# ----------------------------------------------------------
# Load predictions (list of annotations)
# ----------------------------------------------------------
preds = json.loads(pred_json_path.read_text())
print(f"Loaded {len(preds)} polygons.")


# ----------------------------------------------------------
# Helper: unwrap segmentation format
# Some polys are [[[x1,y1,...]]], some [[x1,y1,...]]
# ----------------------------------------------------------
def unwrap(seg):
    s = seg
    while isinstance(s, list) and len(s) > 0 and isinstance(s[0], list):
        s = s[0]
    return s


# ----------------------------------------------------------
# Draw polygons
# ----------------------------------------------------------
for pred in preds:
    seg_flat = unwrap(pred["segmentation"])
    coords = np.array(seg_flat).reshape(-1, 2).astype(float)

    # Draw polygon as a closed loop
    poly_list = coords.flatten().tolist()
    draw.line(poly_list + poly_list[:2], fill=(255, 255, 0), width=3)


# ----------------------------------------------------------
# Visualize
# ----------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.imshow(overlay)
plt.title("PolyWorld (Tiled) Predictions on Original Image")
plt.axis("off")
plt.show()


# ----------------------------------------------------------
# Save overlay
# ----------------------------------------------------------
out_path = pred_json_path.parent / "overlay_tiled_v8.png"
overlay.save(out_path)
print("Saved overlay to:", out_path)


## Run batch processing for all images in test

In [1]:
import os
from pathlib import Path

# import your existing pipeline
from predict_tiled_polyworld import polyworld_inference_and_clean


# ============================================================
# BATCH RUNNER
# ============================================================

def run_batch(
    input_dir,
    output_dir,
    weights_dir,
    iou_thresh=0.20,
    small_thresh=50
):

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    weights_dir = Path(weights_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    # collect only *pre_disaster.png images
    images = sorted(input_dir.glob("*_pre_disaster.png"))
    print(f"[INFO] Found {len(images)} images to process.")

    for img_path in images:
        base = img_path.stem  # e.g. guatemala-volcano_00000003_pre_disaster
        out_json = output_dir / f"{base}.json"

        print(f"\n[INFO] Processing {img_path.name}")
        polyworld_inference_and_clean(
            image_path=str(img_path),
            weights_dir=str(weights_dir),
            output_json=str(out_json),
            iou_thresh=iou_thresh,
            small_thresh=small_thresh,
        )
        print(f"[INFO] Finished {img_path.name}")
    
    print("\n[INFO] Batch processing complete!")


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    run_batch(
        input_dir="/media/data/building_instance_tamu/test/images/",
        output_dir="/media/data/building_instance_tamu/PolyWorld/test_pretrained_scaled/",
        weights_dir="/media/gisense/xihan/250812_tamu_cybertraining_team4/PolyWorld/trained_weights",
        iou_thresh=0.20,
        small_thresh=50,
    )


[INFO] Found 933 images to process.

[INFO] Processing guatemala-volcano_00000003_pre_disaster.png
[INFO] Image size: 1024 × 1024
[INFO] Tile X positions: [0, 320, 640, 960]
[INFO] Tile Y positions: [0, 320, 640, 960]
[INFO] Raw polygons detected: 2
[INFO] After IoU merging: 2 polygons
[INFO] After small-fragment cleaning: 2 polygons
[INFO] Saved cleaned polygons → /media/data/building_instance_tamu/PolyWorld/test_pretrained_scaled/guatemala-volcano_00000003_pre_disaster.json
[INFO] Finished guatemala-volcano_00000003_pre_disaster.png

[INFO] Processing guatemala-volcano_00000005_pre_disaster.png
[INFO] Image size: 1024 × 1024
[INFO] Tile X positions: [0, 320, 640, 960]
[INFO] Tile Y positions: [0, 320, 640, 960]
[INFO] Raw polygons detected: 0
[INFO] After IoU merging: 0 polygons
[INFO] After small-fragment cleaning: 0 polygons
[INFO] Saved cleaned polygons → /media/data/building_instance_tamu/PolyWorld/test_pretrained_scaled/guatemala-volcano_00000005_pre_disaster.json
[INFO] Finishe

## Fine Tuning

In [1]:
# --- PolyWorldTileDatasetV4 ---

import json
import numpy as np
from pathlib import Path
from shapely import wkt as shapely_wkt
from PIL import Image
import torch
from torch.utils.data import Dataset

class PolyWorldTileDatasetV4(Dataset):
    def __init__(self, images_dir, labels_dir, tile=320):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.tile = tile

        self.image_paths = sorted(self.images_dir.glob("*_pre_disaster.png"))
        print(f"[INFO] Found {len(self.image_paths)} images.")

        self.images_cache = []
        for p in self.image_paths:
            self.images_cache.append(np.array(Image.open(p).convert("RGB")))
        print("[INFO] Cached all images.")

        self.polys_cache = []
        for p in self.image_paths:
            jp = self.labels_dir / f"{p.stem}.json"
            polys = []
            if jp.exists():
                ann = json.loads(jp.read_text())
                for feat in ann["features"]["xy"]:
                    try:
                        poly = shapely_wkt.loads(feat["wkt"])
                        polys.append(np.array(poly.exterior.coords, dtype=np.float32))
                    except:
                        continue
            self.polys_cache.append(polys)

        print("[INFO] Cached all polygons.")

        self.tiles = [0, 320, 640, 960]

    def __len__(self):
        return len(self.image_paths) * 16

    def __getitem__(self, idx):
        img_idx = idx // 16
        tile_idx = idx % 16

        img_np = self.images_cache[img_idx]
        polys = self.polys_cache[img_idx]
        H, W = img_np.shape[:2]

        tx = self.tiles[tile_idx % 4]
        ty = self.tiles[tile_idx // 4]

        x2 = min(tx + self.tile, W)
        y2 = min(ty + self.tile, H)
        x1 = max(x2 - self.tile, 0)
        y1 = max(y2 - self.tile, 0)

        patch = img_np[y1:y2, x1:x2]

        if patch.shape[0] != self.tile or patch.shape[1] != self.tile:
            patch = np.array(Image.fromarray(patch).resize((self.tile, self.tile)))

        img_tensor = torch.from_numpy(patch).permute(2, 0, 1).float() / 255.

        # heatmap
        target = torch.zeros((self.tile, self.tile), dtype=torch.float32)

        for poly in polys:
            for vx, vy in poly:
                if x1 <= vx <= x2 and y1 <= vy <= y2:
                    lx = int((vx - x1) / (x2 - x1) * self.tile)
                    ly = int((vy - y1) / (y2 - y1) * self.tile)

                    for dx in range(-2, 3):
                        for dy in range(-2, 3):
                            xx = lx + dx
                            yy = ly + dy
                            if 0 <= xx < self.tile and 0 <= yy < self.tile:
                                target[yy, xx] = 1.0

        return {"image": img_tensor, "target": target.unsqueeze(0)}


In [2]:
from torch.utils.data import DataLoader
import time

loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)

t0 = time.time()
b = next(iter(loader))
print("First batch load time:", time.time() - t0)


NameError: name 'ds' is not defined

In [3]:
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

from models.backbone import R2U_Net, DetectionBranch
from models.matching import OptimalMatching
from models.backbone import NonMaxSuppression


def finetune_polyworld(
    train_images,
    train_labels,
    pretrained_weights,
    output_dir,
    epochs=10,
    lr=1e-4,
    batch_size=8
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ds = PolyWorldTileDatasetV4(train_images, train_labels)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=4)

    # Load pretrained modules
    backbone = R2U_Net().to(device)
    head = DetectionBranch().to(device)

    backbone.load_state_dict(torch.load(pretrained_weights / "polyworld_backbone"))
    head.load_state_dict(torch.load(pretrained_weights / "polyworld_seg_head"))

    # Freeze matching + NMS
    match = OptimalMatching().to(device)
    match.load_state_dict(torch.load(pretrained_weights / "polyworld_matching"))
    for p in match.parameters():
        p.requires_grad = False

    nms = NonMaxSuppression().to(device)
    for p in nms.parameters():
        p.requires_grad = False

    backbone.train()
    head.train()

    optimizer = optim.Adam(
        list(backbone.parameters()) + list(head.parameters()),
        lr=lr
    )
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]).to(device))

    for epoch in range(epochs):
        running = 0
        for batch in loader:
            imgs = batch["image"].to(device)
            tgt = batch["target"].to(device)

            optimizer.zero_grad()
            feat = backbone(imgs)
            pred = head(feat)
            loss = criterion(pred, tgt)

            loss.backward()
            optimizer.step()

            running += loss.item()

        print(f"[Epoch {epoch+1}] loss = {running / len(loader):.6f}")

    # Save
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(backbone.state_dict(), output_dir / "polyworld_backbone_finetuned")
    torch.save(head.state_dict(), output_dir / "polyworld_seg_head_finetuned")
    torch.save(match.state_dict(), output_dir / "polyworld_matching")  # unchanged

    print("✔ Fine-tuning complete.")


In [4]:
finetune_polyworld(
    train_images="/media/data/building_instance_tamu/train/images",
    train_labels="/media/data/building_instance_tamu/train/labels",
    pretrained_weights=Path("/media/gisense/xihan/250812_tamu_cybertraining_team4/PolyWorld/trained_weights"),
    output_dir=Path("/media/data/building_instance_tamu/PolyWorld/finetuned_v2"),
    epochs=20,
    lr=1e-4,
    batch_size=2
)


[INFO] Found 2799 images.
[INFO] Cached all images.
[INFO] Cached all polygons.
[Epoch 1] loss = 0.054521
[Epoch 2] loss = 0.048376
[Epoch 3] loss = 0.044601
[Epoch 4] loss = 0.040852


KeyboardInterrupt: 

In [ ]:
import os
from pathlib import Path

# import your existing pipeline
from predict_tiled_polyworld import polyworld_inference_and_clean


# ============================================================
# BATCH RUNNER
# ============================================================

def run_batch(
    input_dir,
    output_dir,
    weights_dir,
    iou_thresh=0.20,
    small_thresh=50
):

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    weights_dir = Path(weights_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    # collect only *pre_disaster.png images
    images = sorted(input_dir.glob("*_pre_disaster.png"))
    print(f"[INFO] Found {len(images)} images to process.")

    for img_path in images:
        base = img_path.stem  # e.g. guatemala-volcano_00000003_pre_disaster
        out_json = output_dir / f"{base}.json"

        print(f"\n[INFO] Processing {img_path.name}")
        polyworld_inference_and_clean(
            image_path=str(img_path),
            weights_dir=str(weights_dir),
            output_json=str(out_json),
            iou_thresh=iou_thresh,
            small_thresh=small_thresh,
        )
        print(f"[INFO] Finished {img_path.name}")
    
    print("\n[INFO] Batch processing complete!")


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    run_batch(
        input_dir="/media/data/building_instance_tamu/test/images/",
        output_dir="/media/data/building_instance_tamu/PolyWorld/test_finetuned_scaled/",
        weights_dir="/media/data/building_instance_tamu/PolyWorld/finetuned_v2",
        iou_thresh=0.20,
        small_thresh=50,
    )
